# §6 MLP regime 1 / Compare Results — regression, equal wall-clock
Loads `results/{backprop,two_factor,three_factor_clean,three_factor_normal,three_factor_noisy}.json` from Drive (run those first; missing ones skipped). Includes the three-factor cos-sweep head-to-head.

## 1. Setup + Load Results

In [ ]:
import os, json, math, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_DRIVE, DRIVE_SUBDIR = True, 'Section6_regime1'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:', e); STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results'); CKPT_DIR = os.path.join(STORE, 'checkpoints')
METHODS = ['backprop', 'two_factor', 'three_factor_clean', 'three_factor_normal', 'three_factor_noisy']
LABELS = {'backprop':'Backprop', 'two_factor':'Two-factor Hebbian',
          'three_factor_clean':'Three-factor clean (cos~0.5)', 'three_factor_normal':'Three-factor normal (cos~0.09)',
          'three_factor_noisy':'Three-factor noisy (cos~0.01)'}
COLORS = {'backprop':'#1F3864', 'two_factor':'#B8860B', 'three_factor_clean':'#C62828',
          'three_factor_normal':'#E67E22', 'three_factor_noisy':'#7B1FA2'}
R = {}
for m in METHODS:
    p = os.path.join(RESULTS_DIR, f'{m}.json')
    if os.path.exists(p):
        R[m] = json.load(open(p)); s = R[m]['summary']; md = R[m]['meta']
        print(f'loaded {m:24} {md["total_steps"]:>9,} steps  {md["wall_clock_sec"]/3600:5.2f}h  best MSE {s["best_mse"]:.5f}  best R2 {s["best_r2"]:.3f}')
    else:
        print(f'MISSING {p} (run the {m} notebook first)')

## 2. Configuration

In [ ]:
for m in R:
    md = R[m]['meta']; print('='*64); print(LABELS[m]); print(f'  P={md["P"]:,} seed={md["seed"]} samples={md["samples"]:,}'); print('  config:', json.dumps(md['config']))

## 3. Compare curves (MSE + R^2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_r2'],  'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test MSE'); ax[0].set_yscale('log'); ax[0].set_title('Test MSE vs time'); ax[0].legend()
ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('R^2'); ax[1].set_title('Variance explained (R^2) vs time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m.startswith('three_factor'): return st * 2 * R[m]['meta']['config'].get('M', 0)
    if m == 'backprop': return st * 3
    return st * 2
print(f'{"method":30}{"steps":>10}{"fwd-equiv":>14}{"wall h":>8}{"peak MB":>9}')
print('-'*71)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{fwd_equiv(m):>14,}{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## 5. Head-to-head — three-factor cos sweep

In [ ]:
# three-factor cos sweep: clean vs normal vs noisy in equal wall-clock (does more-but-noisier keep winning?)
tf = [m for m in ['three_factor_clean', 'three_factor_normal', 'three_factor_noisy'] if m in R]
if len(tf) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for m in tf:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, c['test_r2'],  'o-', color=COLORS[m], label=LABELS[m])
        ax[2].plot(c['step'], c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_yscale('log'); ax[0].set_xlabel('hours'); ax[0].set_ylabel('test MSE'); ax[0].set_title('cos sweep — MSE vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('R^2'); ax[1].set_title('R^2 vs time'); ax[1].legend()
    ax[2].set_xscale('symlog'); ax[2].set_yscale('log'); ax[2].set_xlabel('steps'); ax[2].set_ylabel('test MSE'); ax[2].set_title('MSE vs steps'); ax[2].legend()
    plt.tight_layout(); plt.show()
    print(f'{"variant":30}{"M":>10}{"cos~":>8}{"steps":>10}{"best_mse":>12}{"best_r2":>10}')
    print('-'*80)
    for m in tf:
        md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0; P = md['P']
        cos = (M/(M+P+1))**0.5 if M else float('nan')
        print(f'{LABELS[m]:30}{M:>10,}{cos:>8.3f}{md["total_steps"]:>10,}{s["best_mse"]:>12.5f}{s["best_r2"]:>10.3f}')
    best = min(tf, key=lambda m: R[m]['summary']['best_mse'])
    print(f'\nLowest best MSE in the budget: {LABELS[best]}. Reading down the cos column shows where fewer '
          f'probes / more steps stops paying off.')
else:
    print('Head-to-head needs >=2 of the three-factor variants (03/04/05).')

## 6. Predicted vs target (best model)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    """Deep tanh MLP. vmap-safe (Linear + tanh only)."""
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        dims = [in_dim] + list(hidden) + [out_dim]
        self.layers = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)])
    def forward(self, x):
        for i, lyr in enumerate(self.layers):
            x = lyr(x)
            if i < len(self.layers) - 1:
                x = torch.tanh(x)
        return x

# predicted vs target for the best (lowest-MSE) model — the regression analog of §8's predictions cell
import torch
if R:
    best = min(R, key=lambda m: R[m]['summary']['best_mse'])
    cfg = R[best]['meta']['config']
    net = MLP(cfg['IN_DIM'], tuple(cfg['HIDDEN']), cfg['OUT_DIM']).to(device)
    ck = torch.load(os.path.join(CKPT_DIR, f'{best}.pt'), map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval()
    torch.manual_seed(R[best]['meta']['seed'])
    teacher = torch.nn.Sequential(torch.nn.Linear(cfg['IN_DIM'],128), torch.nn.Tanh(), torch.nn.Linear(128,cfg['OUT_DIM'])).to(device)
    for p in teacher.parameters(): p.requires_grad_(False)
    x = torch.randn(1024, cfg['IN_DIM'], device=device)
    with torch.no_grad(): pred, tgt = net(x).cpu().flatten(), teacher(x).cpu().flatten()
    plt.figure(figsize=(5,5)); plt.scatter(tgt, pred, s=4, alpha=0.3, color=COLORS[best])
    lim = [min(tgt.min(),pred.min()), max(tgt.max(),pred.max())]; plt.plot(lim, lim, 'k--', lw=1)
    plt.xlabel('teacher target'); plt.ylabel('student prediction'); plt.title(f'Best model: {LABELS[best]} (fit quality)')
    plt.tight_layout(); plt.show()
else:
    print('no results loaded')

## 7. Summary table

In [ ]:
print(f'{"Experiment":30}{"Steps":>10}{"Init MSE":>11}{"Final MSE":>11}{"Best MSE":>11}{"Reduc %":>9}{"Best R2":>9}')
print('-'*91)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{s["initial_mse"]:>11.5f}{s["final_mse"]:>11.5f}{s["best_mse"]:>11.5f}{s["reduction_pct"]:>9.1f}{s["best_r2"]:>9.3f}')